# Análisis Exploratorio de Datos — iFood

**Autores**

- García Juárez Dayan
- Munguía Aguilera Cesar Raúl


El conjunto pertenece a un caso empresarial de CRM y marketing que simula información de clientes.
## Objetivo

Describir la base, evaluar su calidad, estudiar gasto y respuesta a campañas e identificar segmentos útiles para orientar el contacto comercial.

## Contenido

1. Contexto de negocio y unidades monetarias
2. Diccionario de variables
3. Librerías y carga reproducible
4. Inspección inicial y calidad
5. Limpieza y variables derivadas
6. Estadística descriptiva y análisis univariado
7. Relaciones entre variables
8. Productos, dependientes y clientes de alto valor
9. Detección de outliers
10. Antigüedad, clientes tempranos y familia
11. PCA exploratorio
12. Resumen ejecutivo
13. Exportación para el dashboard


## 1. Contexto de negocio y unidades monetarias

La empresa comercializa alimentos mediante tienda física, catálogo y web. El caso busca mejorar campañas directas porque contactar a toda la base genera una respuesta baja.

Los montos se expresan en MU (unidades monetarias genéricas).

Las variables Z_CostContact y Z_Revenue son constantes del caso y permiten estimar el resultado económico del piloto bajo el supuesto de costo por contacto e ingreso por respuesta.


## 2. Diccionario resumido

| Grupo | Variables | Significado |
| :--- | :--- | :--- |
| Identificación | ID | Identificador; se excluye del modelado. |
| Perfil | Year_Birth, Education, Marital_Status, Income | Información demográfica y económica. |
| Hogar | Kidhome, Teenhome | Dependientes en casa. |
| Relación | Dt_Customer, Recency, Complain | Alta, días desde la última compra y quejas. |
| Productos | Variables Mnt | Monto gastado por categoría. |
| Canales | NumWebPurchases, NumCatalogPurchases, NumStorePurchases | Compras por canal. |
| Promociones | NumDealsPurchases, AcceptedCmp1 a AcceptedCmp5 | Compras con descuento e historial de campañas. |
| Actividad | NumWebVisitsMonth | Visitas web en el último mes. |
| Objetivo | Response | Aceptación de la campaña más reciente. |
| Control | Z_CostContact, Z_Revenue | Constantes económicas del caso. |

NumDealsPurchases no se suma a los canales: una compra con descuento también puede realizarse por web, catálogo o tienda, por lo que incluirla duplicaría compras.


## 3. Librerías y carga reproducible

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

DATA_CANDIDATES = [
    Path('data/ifood.csv'),
    Path('../data/ifood.csv')
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA_PATH.resolve()}. '
        'Verifica que exista data/ifood.csv.'
    )

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f'Dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
display(df.head())


## 4. Inspección inicial y calidad

In [ ]:
df.info()

diagnostico = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'porcentaje_nulos': df.isna().mean().mul(100),
    'valores_unicos': df.nunique(dropna=False)
}).sort_values(['nulos', 'valores_unicos'], ascending=[False, True])

duplicados_iniciales = int(df.duplicated().sum())
columnas_constantes = df.columns[df.nunique(dropna=False) == 1].tolist()

print(f'Duplicados: {duplicados_iniciales}')
print(f'Columnas constantes: {columnas_constantes}')
display(diagnostico)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap='viridis', ax=axes[0])
axes[0].set_title('Mapa de valores faltantes')
axes[0].set_xlabel('Variables')

df.nunique(dropna=False).sort_values().plot(kind='bar', color='steelblue', ax=axes[1])
axes[1].set_title('Valores únicos por variable')
axes[1].set_ylabel('Cantidad')
axes[1].tick_params(axis='x', rotation=80)

plt.tight_layout()
plt.show()


### Variable objetivo y economía de la campaña

In [ ]:
contactados = len(df)
respuestas = int(df['Response'].sum())
costo_contacto = float(df['Z_CostContact'].iloc[0])
ingreso_respuesta = float(df['Z_Revenue'].iloc[0])

costo_total = contactados * costo_contacto
ingreso_total = respuestas * ingreso_respuesta
beneficio_neto = ingreso_total - costo_total

resumen_campana = pd.Series({
    'Clientes contactados': contactados,
    'Respuestas positivas': respuestas,
    'Tasa de respuesta (%)': respuestas / contactados * 100,
    'Costo total (MU)': costo_total,
    'Ingreso total (MU)': ingreso_total,
    'Beneficio neto (MU)': beneficio_neto
}, name='Resultado')
display(resumen_campana.to_frame())

conteo = df['Response'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['No respondió', 'Respondió'], conteo.values, color=['lightgray', 'seagreen'])
for i, valor in enumerate(conteo.values):
    ax.text(i, valor + 20, f'{valor:,}\n({valor/contactados:.1%})', ha='center')
ax.set_title('Distribución de Response')
ax.set_ylabel('Clientes')
plt.tight_layout()
plt.show()


La clase positiva es minoritaria. Un modelo futuro debe evaluarse con métricas sensibles al desbalance, como precisión, recall, F1, ROC-AUC y PR-AUC, además de su beneficio económico.


## 5. Limpieza y variables derivadas

In [ ]:
filas_iniciales = len(df)

df = df.drop_duplicates().copy()
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], errors='coerce')
fechas_invalidas = int(df['Dt_Customer'].isna().sum())
df = df.dropna(subset=['Dt_Customer']).copy()

# No existe una fecha oficial de corte; se usa la última alta como referencia interna.
FECHA_REFERENCIA = df['Dt_Customer'].max()
edad_preliminar = FECHA_REFERENCIA.year - df['Year_Birth']
edad_valida = edad_preliminar.between(18, 100)
nacimientos_invalidos = int((~edad_valida).sum())
df = df.loc[edad_valida].copy()

nulos_income_antes = int(df['Income'].isna().sum())
mediana_income = float(df['Income'].median())
df['Income'] = df['Income'].fillna(mediana_income)

print(f'Fecha de referencia interna: {FECHA_REFERENCIA.date()}')
print(f'Duplicados eliminados: {duplicados_iniciales}')
print(f'Fechas inválidas eliminadas: {fechas_invalidas}')
print(f'Nacimientos fuera de rango eliminados: {nacimientos_invalidos}')
print(f'Income imputados con mediana de {mediana_income:,.2f} MU: {nulos_income_antes}')
print(f'Registros: {filas_iniciales:,} -> {len(df):,}')


In [ ]:
columnas_productos = [
    'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds'
]
columnas_canales = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
columnas_campanas = [
    'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
    'AcceptedCmp4', 'AcceptedCmp5'
]

df['Age'] = FECHA_REFERENCIA.year - df['Year_Birth']
df['Customer_Tenure_Days'] = (FECHA_REFERENCIA - df['Dt_Customer']).dt.days
df['Total_Spending'] = df[columnas_productos].sum(axis=1)
df['Total_Purchases'] = df[columnas_canales].sum(axis=1)
df['Average_Ticket'] = (
    df['Total_Spending'] / df['Total_Purchases'].replace(0, np.nan)
)
df['Total_Dependents'] = df['Kidhome'] + df['Teenhome']
df['Accepted_Campaigns_Total'] = df[columnas_campanas].sum(axis=1)
# Metadatos repetidos para distinguir la campaña original de la muestra limpia.
df['Campaign_Total_Contacts'] = filas_iniciales
df['Campaign_Total_Accepted'] = int(df_raw['Response'].sum())

# Response queda fuera de este total para evitar fuga de información.
percentil_90 = float(df['Total_Spending'].quantile(0.90))
df['Value_Segment'] = np.where(
    df['Total_Spending'] >= percentil_90, 'Alto valor', 'Resto'
)

tiene_dependientes = df['Total_Dependents'] > 0
tiene_pareja = df['Marital_Status'].isin(['Married', 'Together'])
df['Family_Segment'] = np.select(
    [
        ~tiene_dependientes,
        tiene_dependientes & tiene_pareja,
        tiene_dependientes & ~tiene_pareja
    ],
    [
        'Sin dependientes',
        'Con dependientes en pareja',
        'Con dependientes sin pareja reportada'
    ],
    default='Sin clasificar'
)

# Variables adicionales requeridas por el dashboard.
df['Income_Segment'] = pd.qcut(
    df['Income'], q=3, labels=['Ingreso bajo', 'Ingreso medio', 'Ingreso alto']
)
df['Spending_Segment'] = pd.qcut(
    df['Total_Spending'], q=4,
    labels=['Gasto bajo', 'Gasto medio-bajo', 'Gasto medio-alto', 'Gasto alto']
)

df['Dependent_Composition'] = np.select(
    [
        (df['Kidhome'] == 0) & (df['Teenhome'] == 0),
        (df['Kidhome'] > 0) & (df['Teenhome'] == 0),
        (df['Kidhome'] == 0) & (df['Teenhome'] > 0),
        (df['Kidhome'] > 0) & (df['Teenhome'] > 0)
    ],
    ['Sin dependientes', 'Con niños', 'Con adolescentes', 'Niños y adolescentes'],
    default='Sin clasificar'
)
df['Marital_Status_Grouped'] = np.select(
    [
        df['Marital_Status'].isin(['Married', 'Together']),
        df['Marital_Status'].isin(['Single', 'Divorced', 'Widow', 'Alone'])
    ],
    ['En pareja', 'Sin pareja reportada'],
    default='Otro / no válido'
)

etiquetas_canales = {
    'NumWebPurchases': 'Web',
    'NumCatalogPurchases': 'Catálogo',
    'NumStorePurchases': 'Tienda'
}
maximo_canal = df[columnas_canales].max(axis=1)
empate_canal = df[columnas_canales].eq(maximo_canal, axis=0).sum(axis=1) > 1
df['Dominant_Channel'] = df[columnas_canales].idxmax(axis=1).map(etiquetas_canales)
df.loc[maximo_canal.eq(0), 'Dominant_Channel'] = 'Sin compras'
df.loc[empate_canal & maximo_canal.gt(0), 'Dominant_Channel'] = 'Empate'
canales_activos = df[columnas_canales].gt(0).sum(axis=1)
df['Channel_Segment'] = np.select(
    [canales_activos.eq(0), canales_activos.eq(1), canales_activos.ge(2)],
    ['Sin compras', 'Monocanal', 'Multicanal'],
    default='Sin clasificar'
)

etiquetas_productos = {
    'MntWines': 'Vinos', 'MntFruits': 'Frutas',
    'MntMeatProducts': 'Carnes', 'MntFishProducts': 'Pescado',
    'MntSweetProducts': 'Dulces', 'MntGoldProds': 'Gold'
}
maximo_producto = df[columnas_productos].max(axis=1)
empate_producto = df[columnas_productos].eq(maximo_producto, axis=0).sum(axis=1) > 1
df['Dominant_Category'] = df[columnas_productos].idxmax(axis=1).map(etiquetas_productos)
df.loc[empate_producto, 'Dominant_Category'] = 'Mixta'
df['Web_Purchase_Visit_Ratio'] = (
    df['NumWebPurchases'] / df['NumWebVisitsMonth'].replace(0, np.nan)
)

variables_creadas = [
    'Age', 'Customer_Tenure_Days', 'Total_Spending', 'Total_Purchases',
    'Average_Ticket', 'Total_Dependents', 'Accepted_Campaigns_Total',
    'Campaign_Total_Contacts', 'Campaign_Total_Accepted',
    'Value_Segment', 'Family_Segment', 'Income_Segment', 'Spending_Segment',
    'Dependent_Composition', 'Marital_Status_Grouped', 'Dominant_Channel',
    'Channel_Segment', 'Dominant_Category', 'Web_Purchase_Visit_Ratio'
]
display(df[variables_creadas].head())


In [ ]:
validacion = pd.Series({
    'Filas finales': len(df),
    'Duplicados finales': int(df.duplicated().sum()),
    'Nulos pendientes en variables originales': int(df[df_raw.columns].isna().sum().sum()),
    'Tickets no definidos por 0 compras': int(df['Average_Ticket'].isna().sum()),
    'Edad mínima': int(df['Age'].min()),
    'Edad máxima': int(df['Age'].max()),
    'Antigüedad mínima (días)': int(df['Customer_Tenure_Days'].min()),
    'Antigüedad máxima (días)': int(df['Customer_Tenure_Days'].max())
}, name='Valor')
display(validacion.to_frame())


### Impacto visual de la limpieza

La comparación siguiente verifica cuántos registros se conservaron y si permanecen valores faltantes en las variables originales. Los tickets sin compras se analizan aparte, porque son valores derivados no definidos y no datos originales ausentes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

conteos_limpieza = {
    'Antes de limpiar': len(df_raw),
    'Después de limpiar': len(df)
}
barras = axes[0].bar(
    conteos_limpieza.keys(),
    conteos_limpieza.values(),
    color=['salmon', 'lightseagreen']
)
for barra in barras:
    altura = barra.get_height()
    axes[0].text(
        barra.get_x() + barra.get_width() / 2,
        altura + 8,
        f'{int(altura):,}',
        ha='center',
        fontweight='bold'
    )
axes[0].set_title('Registros antes y después de la limpieza')
axes[0].set_ylabel('Clientes')
axes[0].set_ylim(0, max(conteos_limpieza.values()) * 1.08)

sns.heatmap(
    df[df_raw.columns].isna(),
    cbar=False,
    yticklabels=False,
    cmap='viridis',
    ax=axes[1]
)
axes[1].set_title('Nulos restantes en variables originales')
axes[1].set_xlabel('Variables')

plt.show()


La edad queda alineada con el periodo de los datos y no con el año actual. La antigüedad es relativa porque la base no proporciona una fecha oficial de cierre de campaña.


## 6. Estadística descriptiva y análisis univariado

In [ ]:
variables_clave = [
    'Income', 'Age', 'Customer_Tenure_Days', 'Recency',
    'Total_Spending', 'Total_Purchases', 'Average_Ticket',
    'NumWebVisitsMonth', 'Accepted_Campaigns_Total'
]
estadisticas = df[variables_clave].describe().T
estadisticas['mediana'] = df[variables_clave].median()
estadisticas['asimetria'] = df[variables_clave].skew()
display(estadisticas[['count', 'mean', 'mediana', 'std', 'min', 'max', 'asimetria']])


### Lectura de las estadísticas descriptivas

- Income presenta un extremo muy alto: la mediana es más representativa que la media para describir al hogar típico.
- Total_Spending tiene asimetría positiva; la media supera claramente a la mediana porque un grupo pequeño concentra gasto elevado.
- La edad se calcula en el periodo histórico de los datos, no en 2026, por lo que no debe compararse con edades actuales.
- Recency cubre de manera casi uniforme el rango de 0 a 99 días.
- La actividad web típica se sitúa alrededor de cinco o seis visitas mensuales.

Estas observaciones son descriptivas y no implican diferencias estadísticamente significativas.


In [ ]:
variables_distribucion = ['Income', 'Age', 'Total_Spending', 'Recency']
fig, axes = plt.subplots(len(variables_distribucion), 2, figsize=(13, 14))

for i, columna in enumerate(variables_distribucion):
    sns.histplot(df[columna], kde=True, color='steelblue', ax=axes[i, 0])
    axes[i, 0].set_title(f'Distribución de {columna}')
    sns.boxplot(x=df[columna], color='salmon', ax=axes[i, 1])
    axes[i, 1].set_title(f'Dispersión de {columna}')

plt.tight_layout()
plt.show()


### Distribuciones de productos y actividad digital

Además de las variables generales, se revisan dos categorías de alto peso comercial y la actividad web. Esto permite distinguir entre una distribución de gasto con cola derecha y una variable de interacción digital más compacta.


In [ ]:
variables_detalle = ['MntWines', 'MntMeatProducts', 'NumWebVisitsMonth']
fig, axes = plt.subplots(len(variables_detalle), 2, figsize=(13, 10))

for i, columna in enumerate(variables_detalle):
    sns.histplot(df[columna], kde=True, color='skyblue', ax=axes[i, 0])
    axes[i, 0].set_title(f'Distribución de {columna}')
    sns.boxplot(x=df[columna], color='lightseagreen', ax=axes[i, 1])
    axes[i, 1].set_title(f'Dispersión y outliers de {columna}')

plt.tight_layout()
plt.show()


Los gastos en vinos y carnes muestran colas hacia valores altos: la mayoría compra poco o moderadamente, mientras algunos clientes concentran montos elevados. NumWebVisitsMonth es más compacto, aunque conserva algunos casos extremos. Un outlier comercial no se considera automáticamente un error.


### Educación y estado civil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(
    data=df, x='Education',
    order=df['Education'].value_counts().index,
    color='skyblue', ax=axes[0]
)
axes[0].set_title('Nivel educativo')
axes[0].tick_params(axis='x', rotation=35)

sns.countplot(
    data=df, x='Marital_Status',
    order=df['Marital_Status'].value_counts().index,
    color='lightgreen', ax=axes[1]
)
axes[1].set_title('Estado civil reportado')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


## 7. Relaciones entre variables

In [ ]:
variables_correlacion = [
    'Income', 'Age', 'Customer_Tenure_Days', 'Recency',
    'Total_Spending', 'Total_Purchases', 'Average_Ticket',
    'Total_Dependents', 'NumWebVisitsMonth',
    'Accepted_Campaigns_Total', 'Complain', 'Response'
]
matriz_correlacion = df[variables_correlacion].corr()
mask = np.triu(np.ones_like(matriz_correlacion, dtype=bool))

plt.figure(figsize=(13, 9))
sns.heatmap(
    matriz_correlacion, mask=mask, cmap='RdYlGn',
    center=0, annot=True, fmt='.2f'
)
plt.title('Matriz de correlación de Pearson')
plt.tight_layout()
plt.show()

display(
    matriz_correlacion['Response']
    .drop('Response')
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .to_frame('correlacion_con_Response')
)


In [ ]:
correlacion_ingreso_vino = df['Income'].corr(df['MntWines'])
print(f'Correlación Income–MntWines: {correlacion_ingreso_vino:.3f}')

plt.figure(figsize=(9, 6))
sns.regplot(
    data=df, x='Income', y='MntWines',
    scatter_kws={'alpha': 0.35, 's': 25},
    line_kws={'color': 'firebrick'}
)
plt.title('Ingreso y gasto en vinos')
plt.xlabel('Ingreso (MU)')
plt.ylabel('Gasto en vinos (MU)')
plt.tight_layout()
plt.show()


La correlación representa asociación lineal, no causalidad. El ingreso extremo debe considerarse al interpretar la escala.


### Canales y respuesta a la campaña

In [ ]:
resumen_canales = df.groupby('Response')[columnas_canales].agg(['mean', 'median'])
display(resumen_canales)

canales_largos = df.melt(
    id_vars='Response',
    value_vars=columnas_canales,
    var_name='Canal',
    value_name='Compras'
)
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=canales_largos, x='Canal', y='Compras',
    hue='Response', palette='Set2'
)
plt.title('Compras por canal según Response')
plt.tight_layout()
plt.show()


Las diferencias son descriptivas. No se califican como estadísticamente significativas porque no se aplicaron pruebas de hipótesis.


## 8. Productos, dependientes y clientes de alto valor

In [ ]:
resumen_productos = pd.DataFrame({
    'promedio_MU': df[columnas_productos].mean(),
    'mediana_MU': df[columnas_productos].median(),
    'participacion_pct': (
        df[columnas_productos].sum()
        / df[columnas_productos].to_numpy().sum() * 100
    )
}).sort_values('promedio_MU', ascending=False)
display(resumen_productos)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
resumen_productos['promedio_MU'].plot(kind='bar', color='firebrick', ax=axes[0])
axes[0].set_title('Gasto promedio por producto')
axes[0].set_ylabel('MU')
resumen_productos['participacion_pct'].plot(kind='bar', color='teal', ax=axes[1])
axes[1].set_title('Participación en el gasto')
axes[1].set_ylabel('%')
for ax in axes:
    ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()


In [ ]:
resumen_dependientes = df.groupby('Total_Dependents')['Total_Spending'].agg(
    clientes='size', media='mean', mediana='median'
)
display(resumen_dependientes)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Total_Dependents', y='Total_Spending', color='plum')
plt.title('Gasto total según dependientes')
plt.ylabel('Gasto total (MU)')
plt.tight_layout()
plt.show()


### Gasto en carnes según dependientes

El gasto total y el gasto por producto no necesariamente siguen el mismo patrón. Se añade una inspección específica de carnes para evaluar si su distribución cambia con la composición del hogar.


In [ ]:
resumen_carne_dependientes = df.groupby('Total_Dependents')['MntMeatProducts'].agg(
    clientes='size',
    gasto_promedio='mean',
    gasto_mediano='median'
)
display(resumen_carne_dependientes)

plt.figure(figsize=(9, 6))
sns.violinplot(
    data=df,
    x='Total_Dependents',
    y='MntMeatProducts',
    color='lightcoral',
    inner='quartile',
    cut=0
)
plt.title('Gasto en carnes según dependientes en el hogar')
plt.xlabel('Número de dependientes')
plt.ylabel('Gasto en carnes (MU)')
plt.tight_layout()
plt.show()


La distribución es asimétrica a la derecha. Los hogares sin dependientes presentan un gasto típico en carnes considerablemente mayor, aunque existen clientes de gasto alto en distintos tipos de hogar. La gráfica describe asociación, no un efecto causal de los dependientes.


In [ ]:
resumen_valor = df.groupby('Value_Segment', observed=True).agg(
    clientes=('ID', 'size'),
    gasto_total_MU=('Total_Spending', 'sum'),
    gasto_promedio_MU=('Total_Spending', 'mean'),
    ingreso_mediano_MU=('Income', 'median'),
    compras_medianas=('Total_Purchases', 'median'),
    tasa_respuesta=('Response', 'mean')
)
resumen_valor['participacion_clientes_pct'] = resumen_valor['clientes'] / len(df) * 100
resumen_valor['participacion_gasto_pct'] = (
    resumen_valor['gasto_total_MU'] / df['Total_Spending'].sum() * 100
)
resumen_valor['tasa_respuesta'] *= 100

print(f'Umbral P90 de alto valor: {percentil_90:,.2f} MU')
display(resumen_valor)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df, x='Value_Segment', y='Income', color='lightblue', ax=axes[0])
axes[0].set_title('Ingreso por segmento')
axes[0].set_ylabel('Ingreso (MU)')
sns.boxplot(data=df, x='Value_Segment', y='Total_Purchases', color='lightgreen', ax=axes[1])
axes[1].set_title('Compras por segmento')
plt.tight_layout()
plt.show()


In [ ]:
proporciones = df[columnas_productos].div(
    df['Total_Spending'].replace(0, np.nan), axis=0
)
proporciones['Value_Segment'] = df['Value_Segment']
composicion_segmento = proporciones.groupby('Value_Segment', observed=True).mean()

plt.figure(figsize=(8, 6))
sns.heatmap(composicion_segmento.T, annot=True, cmap='YlGnBu', fmt='.1%')
plt.title('Composición del gasto por segmento')
plt.tight_layout()
plt.show()
display(composicion_segmento)


## 9. Detección de outliers mediante IQR

In [ ]:
def resumen_outliers_iqr(dataframe, columnas):
    registros = []
    for columna in columnas:
        q1 = dataframe[columna].quantile(0.25)
        q3 = dataframe[columna].quantile(0.75)
        iqr = q3 - q1
        inferior = q1 - 1.5 * iqr
        superior = q3 + 1.5 * iqr
        mascara = (dataframe[columna] < inferior) | (dataframe[columna] > superior)
        registros.append({
            'variable': columna,
            'limite_inferior': inferior,
            'limite_superior': superior,
            'outliers': int(mascara.sum()),
            'porcentaje': mascara.mean() * 100
        })
    return pd.DataFrame(registros).set_index('variable')

columnas_outliers = [
    'Income', 'Age', 'Recency',
    *columnas_productos,
    'NumDealsPurchases', *columnas_canales, 'NumWebVisitsMonth',
    'Total_Spending', 'Total_Purchases', 'Average_Ticket'
]
tabla_outliers = resumen_outliers_iqr(df, columnas_outliers)
display(tabla_outliers.sort_values('outliers', ascending=False))

columnas_outliers_grafica = [
    'Income', 'Age', 'Total_Spending', 'Total_Purchases', 'Average_Ticket'
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, columna in enumerate(columnas_outliers_grafica):
    sns.boxplot(x=df[columna], color='salmon', ax=axes[i])
    axes[i].set_title(columna)
axes[-1].axis('off')
plt.tight_layout()
plt.show()


Los outliers comerciales no se eliminan automáticamente: algunos representan clientes genuinos de alto valor. Solo se retiraron edades incompatibles con el periodo observado.


## 10. Antigüedad, clientes tempranos y familia

In [ ]:
orden_antiguedad = ['Nuevo', 'Reciente', 'Estable', 'Leal']
df['Tenure_Group'] = pd.qcut(
    df['Customer_Tenure_Days'], q=4, labels=orden_antiguedad
)

# Cohortes estratégicas para el dashboard. Pueden solaparse entre sí.
umbral_gasto_p95 = float(df['Total_Spending'].quantile(0.95))
umbral_visitas_p75 = float(df['NumWebVisitsMonth'].quantile(0.75))
umbral_compras_web_p25 = float(df['NumWebPurchases'].quantile(0.25))
umbral_descuentos_p75 = float(df['NumDealsPurchases'].quantile(0.75))
umbral_recencia_p75 = float(df['Recency'].quantile(0.75))
mediana_gasto = float(df['Total_Spending'].median())

df['Is_Premium'] = df['Value_Segment'].eq('Alto valor')
df['Is_High_Income_Low_Spending'] = (
    df['Income_Segment'].eq('Ingreso alto')
    & df['Total_Spending'].le(mediana_gasto)
)
df['Is_Multichannel'] = df['Channel_Segment'].eq('Multicanal')
df['Is_Web_Curious'] = (
    df['NumWebVisitsMonth'].ge(umbral_visitas_p75)
    & df['NumWebPurchases'].le(umbral_compras_web_p25)
)
df['Is_Discount_Sensitive'] = df['NumDealsPurchases'].ge(umbral_descuentos_p75)
df['Is_Loyal'] = df['Tenure_Group'].eq('Leal')
df['Is_Early_High_Value'] = (
    df['Tenure_Group'].isin(['Nuevo', 'Reciente'])
    & df['Total_Spending'].ge(umbral_gasto_p95)
)
df['Is_Cold'] = df['Recency'].ge(umbral_recencia_p75)

reglas_segmentos = [
    ('Nuevo de alto valor', 'Is_Early_High_Value'),
    ('Premium', 'Is_Premium'),
    ('Alto ingreso, bajo gasto', 'Is_High_Income_Low_Spending'),
    ('Web curioso', 'Is_Web_Curious'),
    ('Inactivo o frío', 'Is_Cold'),
    ('Leal', 'Is_Loyal'),
    ('Sensible a descuentos', 'Is_Discount_Sensitive'),
    ('Multicanal', 'Is_Multichannel')
]
df['Strategic_Tags'] = df.apply(
    lambda fila: ' | '.join(
        nombre for nombre, columna in reglas_segmentos if fila[columna]
    ) or 'Base general',
    axis=1
)
df['Primary_Strategic_Segment'] = np.select(
    [df[columna] for _, columna in reglas_segmentos],
    [nombre for nombre, _ in reglas_segmentos],
    default='Base general'
)

resumen_antiguedad = df.groupby('Tenure_Group', observed=True).agg(
    clientes=('ID', 'size'),
    dias_minimos=('Customer_Tenure_Days', 'min'),
    dias_maximos=('Customer_Tenure_Days', 'max'),
    gasto_mediano=('Total_Spending', 'median'),
    tasa_respuesta=('Response', 'mean')
)
resumen_antiguedad['tasa_respuesta'] *= 100
display(resumen_antiguedad)

plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df, x='Tenure_Group', y='Total_Spending',
    order=orden_antiguedad, color='mediumseagreen'
)
plt.title('Gasto por cuartil relativo de antigüedad')
plt.ylabel('Gasto total (MU)')
plt.tight_layout()
plt.show()


### Relación continua entre antigüedad y gasto

Los cuartiles resumen diferencias entre grupos, pero pueden ocultar la dispersión interna. La siguiente gráfica conserva los días de antigüedad y distingue a los clientes de alto valor.


In [ ]:
correlacion_antiguedad_gasto = df['Customer_Tenure_Days'].corr(df['Total_Spending'])
print(
    'Correlación de Pearson entre antigüedad y gasto: '
    f'{correlacion_antiguedad_gasto:.3f}'
)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='Customer_Tenure_Days',
    y='Total_Spending',
    hue='Value_Segment',
    palette={'Resto': 'lightgray', 'Alto valor': 'firebrick'},
    alpha=0.6
)
sns.regplot(
    data=df,
    x='Customer_Tenure_Days',
    y='Total_Spending',
    scatter=False,
    color='black',
    line_kws={'linewidth': 1.5}
)
plt.title('Antigüedad y gasto total')
plt.xlabel('Antigüedad relativa (días)')
plt.ylabel('Gasto total (MU)')
plt.tight_layout()
plt.show()


La relación lineal es positiva pero débil: la antigüedad por sí sola explica poco de la variación del gasto. Esto es compatible con el aumento de medianas por cuartil, pero también muestra clientes de gasto alto en distintas etapas de la relación.


In [ ]:
umbral_vip_temprano = float(df['Total_Spending'].quantile(0.95))
clientes_vip_tempranos = df[
    df['Tenure_Group'].isin(['Nuevo', 'Reciente'])
    & (df['Total_Spending'] >= umbral_vip_temprano)
].copy()

print(f'VIP en los dos cuartiles de menor antigüedad: {len(clientes_vip_tempranos)}')
print(f'Umbral P95: {umbral_vip_temprano:,.2f} MU')
display(
    clientes_vip_tempranos[
        ['Age', 'Income', 'Total_Spending', 'Total_Purchases', 'Customer_Tenure_Days']
    ].agg(['mean', 'median']).T
)
display(clientes_vip_tempranos[columnas_productos].mean().to_frame('promedio_MU'))


In [ ]:
resumen_familiar = df.groupby('Family_Segment').agg(
    clientes=('ID', 'size'),
    gasto_promedio=('Total_Spending', 'mean'),
    gasto_mediano=('Total_Spending', 'median'),
    ingreso_mediano=('Income', 'median'),
    tasa_respuesta=('Response', 'mean')
).sort_values('gasto_mediano', ascending=False)
resumen_familiar['tasa_respuesta'] *= 100
display(resumen_familiar)

plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x='Family_Segment', y='Total_Spending', color='wheat')
plt.title('Gasto por segmento familiar')
plt.ylabel('Gasto total (MU)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 11. PCA exploratorio

El PCA resume las relaciones numéricas, pero no crea segmentos por sí solo. Para reducir el efecto de extremos, las variables se limitan temporalmente a sus percentiles 1 y 99; el dataframe original no se modifica.


In [ ]:
columnas_pca = [
    'Income', 'Total_Spending', 'Age',
    'Customer_Tenure_Days', 'Recency', 'NumWebVisitsMonth'
]
pca_input = df[columnas_pca].copy()
pca_limitado = pca_input.clip(
    lower=pca_input.quantile(0.01),
    upper=pca_input.quantile(0.99),
    axis=1
)
desviaciones = pca_limitado.std(ddof=0).replace(0, 1)
pca_z = (pca_limitado - pca_limitado.mean()) / desviaciones

covarianza = np.cov(pca_z.to_numpy(), rowvar=False, bias=True)
autovalores, autovectores = np.linalg.eigh(covarianza)
orden = np.argsort(autovalores)[::-1]
autovalores = autovalores[orden]
autovectores = autovectores[:, orden]
varianza_explicada = autovalores / autovalores.sum()

componentes = pca_z.to_numpy() @ autovectores[:, :2]
pca_df = pd.DataFrame(componentes, columns=['PC1', 'PC2'], index=df.index)
df['PCA_PC1'] = pca_df['PC1']
df['PCA_PC2'] = pca_df['PC2']
pca_df['Value_Segment'] = df['Value_Segment']

cargas = pd.DataFrame(
    autovectores[:, :2], index=columnas_pca, columns=['PC1', 'PC2']
)
varianza = pd.Series(
    varianza_explicada[:2], index=['PC1', 'PC2'], name='varianza_explicada'
)
display(varianza.mul(100).to_frame('varianza_explicada_pct'))
display(cargas)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2', hue='Value_Segment',
    alpha=0.55,
    palette={'Resto': 'lightgray', 'Alto valor': 'firebrick'}
)
plt.title(f'PCA — varianza acumulada: {varianza_explicada[:2].sum():.1%}')
plt.tight_layout()
plt.show()


## 12. Resumen ejecutivo

In [ ]:
alto_valor = resumen_valor.loc['Alto valor']

indicadores = pd.Series({
    'Clientes analizados': len(df),
    'Tasa de respuesta (%)': df['Response'].mean() * 100,
    'Gasto mediano (MU)': df['Total_Spending'].median(),
    'Gasto promedio (MU)': df['Total_Spending'].mean(),
    'Umbral de alto valor P90 (MU)': percentil_90,
    'Clientes de alto valor (%)': alto_valor['participacion_clientes_pct'],
    'Gasto concentrado en alto valor (%)': alto_valor['participacion_gasto_pct'],
    'Respuesta del segmento de alto valor (%)': alto_valor['tasa_respuesta'],
    'Beneficio del contacto masivo (MU)': beneficio_neto
}, name='Valor')
display(indicadores.to_frame())


### Hallazgos

1. La respuesta es minoritaria y el contacto masivo produce un resultado económico negativo bajo los supuestos del caso.
2. El gasto presenta asimetría positiva; la mediana representa mejor al cliente típico que la media.
3. El segmento de alto valor debe evaluarse tanto por gasto como por respuesta.
4. Canales, antigüedad y familia muestran asociaciones descriptivas, no relaciones causales.
5. Los extremos comerciales pueden ser clientes valiosos y requieren investigación antes de eliminarlos.

### Recomendaciones

- Construir un modelo de propensión para Response con validación estratificada y métricas para clases desbalanceadas.
- Excluir ID, las columnas constantes y cualquier variable que genere fuga de información.
- Comparar el beneficio esperado del modelo contra el contacto masivo.
- Mantener los montos etiquetados como MU mientras no exista una moneda documentada.

### Limitaciones

- El dataset corresponde a un caso simulado.
- No existe una fecha oficial de corte; se usa la última fecha de alta como referencia interna.
- La imputación por mediana introduce una suposición.
- Este análisis es exploratorio y no establece causalidad.


## 13. Exportación para el dashboard

El dashboard utilizará exclusivamente el resultado de este notebook. Cada vez que cambien las reglas de limpieza o las variables derivadas, se debe ejecutar el notebook completo para regenerar el archivo.

Para cubrir la navegación y los filtros del dashboard se añadieron variables que no formaban parte del primer EDA: segmentos de ingreso y gasto, composición de dependientes, estado civil agrupado, canal y categoría dominantes, perfil de canales, razón aproximada de compras/visitas web, etiquetas de segmentos estratégicos y las dos coordenadas del PCA. Las reglas completas y sus limitaciones se documentan en `DASHBOARD.md`.


In [ ]:
OUTPUT_CANDIDATES = [
    Path('data/ifood_dashboard.csv'),
    Path('../data/ifood_dashboard.csv')
]
OUTPUT_PATH = next(
    (path for path in OUTPUT_CANDIDATES if path.parent.exists()),
    OUTPUT_CANDIDATES[0]
)

df.to_csv(OUTPUT_PATH, index=False)

print(f'Archivo para el dashboard: {OUTPUT_PATH.resolve()}')
print(f'Dimensiones exportadas: {df.shape[0]:,} filas × {df.shape[1]} columnas')
